# Recommender System

By Tarik Kalai and Joaquín Roa

May 31st 2026

We will construct a recommender system for games on Steam using LLMs.

In [11]:
import pandas as pd
import numpy as np
import sqlite3

In [12]:
# pip install pandas numpy sqlite3

In [13]:
# DB_PATH = "C:/Users/u0181444/OneDrive - KU Leuven/Desktop/2602 Advanced Analytics in Business/assignments/recommender_system/raglooker/steam_games_reviews_25.sqlite"
DB_PATH = "steam_games_reviews_25.sqlite"

In [14]:
# "steam_games_reviews_25.sqlite"   # adjust if needed
con = sqlite3.connect(DB_PATH)

# 1. What tables exist?
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))

      name
0    games
1  reviews


In [15]:
# Row counts
n_games   = pd.read_sql("SELECT COUNT(*) AS n FROM games",   con).iloc[0, 0]
n_reviews = pd.read_sql("SELECT COUNT(*) AS n FROM reviews", con).iloc[0, 0]
print(f"games: {n_games:,} | reviews: {n_reviews:,}")

games: 39,176 | reviews: 7,679,845


In [16]:
games = pd.read_sql("SELECT * FROM games", con)
print(games.shape)
print(games.dtypes)
games.head()

(39176, 44)
appid                          int64
name                             str
release_date                     str
required_age                   int64
price                        float64
dlc_count                      int64
detailed_description             str
about_the_game                   str
short_description                str
reviews_summary                  str
header_image                     str
website                          str
support_url                      str
support_email                    str
windows                        int64
mac                            int64
linux                          int64
metacritic_score               int64
metacritic_url                   str
achievements                   int64
recommendations                int64
notes                            str
supported_languages_json         str
full_audio_languages_json        str
packages_json                    str
developers_json                  str
publishers_json           

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews_summary,...,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags_json,raw_json
0,10,Counter-Strike,"Nov 1, 2000",0,9.99,0,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,,...,6427,10000000 - 20000000,11335,508,175,68,0,7323,"{""Action"": 5504, ""FPS"": 4929, ""Multiplayer"": 3...","{""name"": ""Counter-Strike"", ""release_date"": ""No..."
1,20,Team Fortress Classic,"Apr 1, 1999",0,4.99,0,One of the most popular online action games of...,One of the most popular online action games of...,One of the most popular online action games of...,,...,1136,1000000 - 2000000,666,0,13,0,0,66,"{""Action"": 767, ""FPS"": 333, ""Multiplayer"": 280...","{""name"": ""Team Fortress Classic"", ""release_dat..."
2,30,Day of Defeat,"May 1, 2003",0,4.99,0,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,,...,688,5000000 - 10000000,6526,0,18,0,0,87,"{""FPS"": 804, ""World War II"": 272, ""Multiplayer...","{""name"": ""Day of Defeat"", ""release_date"": ""May..."
3,40,Deathmatch Classic,"Jun 1, 2001",0,4.99,0,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...,,...,545,5000000 - 10000000,151,0,7,0,0,7,"{""Action"": 638, ""FPS"": 155, ""Classic"": 119, ""M...","{""name"": ""Deathmatch Classic"", ""release_date"":..."
4,50,Half-Life: Opposing Force,"Nov 1, 1999",0,4.99,0,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,,...,1198,2000000 - 5000000,406,0,183,0,0,74,"{""FPS"": 934, ""Action"": 362, ""Classic"": 291, ""S...","{""name"": ""Half-Life: Opposing Force"", ""release..."


In [17]:
reviews = pd.read_sql("SELECT * FROM reviews LIMIT 5000", con)
print(reviews.shape)
print(reviews.dtypes)
reviews.head()

(5000, 21)
recommendationid                 str
appid                          int64
language                         str
review                           str
timestamp_created              int64
timestamp_updated              int64
voted_up                       int64
votes_up                       int64
votes_funny                    int64
weighted_vote_score              str
comment_count                  int64
steam_purchase                 int64
received_for_free              int64
written_during_early_access    int64
primarily_steam_deck           int64
refunded                       int64
app_release_date               int64
author_steamid                   str
author_json                      str
reactions_json                   str
raw_json                         str
dtype: object


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,steam_purchase,received_for_free,written_during_early_access,primarily_steam_deck,refunded,app_release_date,author_steamid,author_json,reactions_json,raw_json
0,223036085,10,english,better than cs2\r\n,1775926201,1775926201,1,0,0,0.5,...,1,0,0,0,0,973065600,76561199016330204,"{""steamid"": ""76561199016330204"", ""personaname""...",[],"{""recommendationid"": ""223036085"", ""author"": {""..."
1,223015211,10,english,amazing,1775908677,1775908677,1,0,0,0.5,...,0,0,0,0,0,973065600,76561199466466829,"{""steamid"": ""76561199466466829"", ""personaname""...",[],"{""recommendationid"": ""223015211"", ""author"": {""..."
2,222991906,10,english,legend for 2008,1775879493,1775879493,1,0,0,0.5,...,1,0,0,0,0,973065600,76561198782202751,"{""steamid"": ""76561198782202751"", ""personaname""...",[],"{""recommendationid"": ""222991906"", ""author"": {""..."
3,222984439,10,english,Extremely simple game and yet it is also extre...,1775869746,1775869746,1,0,0,0.5,...,1,0,0,0,0,973065600,76561199036635082,"{""steamid"": ""76561199036635082"", ""personaname""...",[],"{""recommendationid"": ""222984439"", ""author"": {""..."
4,222970777,10,english,LEGEND GAME,1775853458,1775853458,1,0,0,0.5,...,1,0,0,0,0,973065600,76561199575900532,"{""steamid"": ""76561199575900532"", ""personaname""...",[],"{""recommendationid"": ""222970777"", ""author"": {""..."


In [18]:
# Pull top-10 reviews per game by weighted_vote_score (English only, non-empty)
digests = pd.read_sql("""
    WITH ranked AS (
        SELECT appid, review,
               ROW_NUMBER() OVER (
                   PARTITION BY appid
                   ORDER BY weighted_vote_score DESC, votes_up DESC
               ) AS rn
        FROM reviews
        WHERE language = 'english'
          AND review IS NOT NULL
          AND LENGTH(review) BETWEEN 30 AND 1500
    )
    SELECT appid, GROUP_CONCAT(review, '  ||  ') AS review_digest
    FROM ranked
    WHERE rn <= 10
    GROUP BY appid
""", con)

print(digests.shape)            # expect ~39k rows
print(digests["review_digest"].str.len().describe())
digests.head(2)

(39271, 2)
count    39271.000000
mean      4544.764254
std       1724.690374
min         62.000000
25%       3254.000000
50%       4434.000000
75%       5706.000000
max      11826.000000
Name: review_digest, dtype: float64


,appid,review_digest
0,10,---{ Graphics }---\n☐ You forget what reality ...
1,20,Since no one on TF community will read this re...


In [19]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

# Prefix per nomic's convention
docs = ("search_document: " + digests["review_digest"]).tolist()
ids  = digests["appid"].astype(str).tolist()

# Embed in batches (progress bar shown). CPU is fine; ~10-20 min for 39k.
embeddings = model.encode(
    docs,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # so cosine == dot product
)

print(embeddings.shape)  # (39271, 768)

<All keys matched successfully>


Batches:   0%|          | 0/1228 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


KeyboardInterrupt: 

In [ ]:
# import sys
# print(sys.executable)

c:\Users\u0181444\OneDrive - KU Leuven\Desktop\2602 Advanced Analytics in Business\assignments\recommender_system\raglooker\.venv\Scripts\python.exe


In [20]:
digests = pd.read_sql("""
    WITH ranked AS (
        SELECT appid, review,
               ROW_NUMBER() OVER (
                   PARTITION BY appid
                   ORDER BY weighted_vote_score DESC, votes_up DESC
               ) AS rn
        FROM reviews
        WHERE language = 'english'
          AND review IS NOT NULL
          AND LENGTH(review) BETWEEN 30 AND 400
    )
    SELECT appid, GROUP_CONCAT(review, ' || ') AS review_digest
    FROM ranked
    WHERE rn <= 4
    GROUP BY appid
""", con)

# Hard cap to be safe
digests["review_digest"] = digests["review_digest"].str.slice(0, 900)
print(digests.shape, digests["review_digest"].str.len().describe())

(39271, 2) count    39271.000000
mean       721.472817
std        181.974354
min         62.000000
25%        597.000000
50%        765.000000
75%        900.000000
max        900.000000
Name: review_digest, dtype: float64


In [21]:
from sentence_transformers import SentenceTransformer
import chromadb, numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# No prefix needed for MiniLM — simpler.

docs = digests["review_digest"].tolist()
ids  = digests["appid"].astype(str).tolist()

embeddings = model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(embeddings.shape)   # (~39k, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\u0181444\OneDrive - KU Leuven\Desktop\2602 Advanced Analytics in Business\assignments\recommender_system\raglooker\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\u0181444\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(messag

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/614 [00:00<?, ?it/s]

(39271, 384)


In [22]:
client = chromadb.PersistentClient(path="./chroma_db")
coll = client.get_or_create_collection(
    name="steam_games",
    metadata={"hnsw:space": "cosine"},
)

# Add in chunks (Chroma's add() has a per-call size limit)
B = 1000
for i in range(0, len(ids), B):
    coll.add(
        ids=ids[i:i+B],
        embeddings=embeddings[i:i+B].tolist(),
    )

print(coll.count())

39271


In [23]:
q = "chill cozy farming game with co-op"
q_emb = model.encode([q], normalize_embeddings=True)

res = coll.query(query_embeddings=q_emb.tolist(), n_results=5)
for appid, dist in zip(res["ids"][0], res["distances"][0]):
    name = games.loc[games["appid"].astype(str) == appid, "name"].iloc[0]
    print(f"{1 - dist:.3f}  {appid:>8}  {name}")

0.628   3837530  Magical Greenhouse
0.606   1358790  Tipston Salvage
0.602   1977530  One-armed cook
0.600    303260  TRISTOY
0.598   1262460  Zompiercer


In [24]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")  # use the SAME path you used
print("Collections:", [c.name for c in client.list_collections()])
for c in client.list_collections():
    coll = client.get_collection(c.name)
    print(f"  {c.name}: {coll.count()} items")

Collections: ['steam_games']
  steam_games: 39271 items


In [25]:
import ollama

resp = ollama.chat(
    model="gemma4:latest",   
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)
print(resp["message"]["content"])

Hello there!


Kind of Obi-Wan Kenobi-coded but OK.